In [ ]:
from pynq import Overlay, MMIO, allocate
import numpy as np
from numpy.lib.stride_tricks import as_strided
import cv2
import time
from collections import deque
import os
from PIL import Image
import matplotlib.pyplot as plt
import math
from math import ceil
from typing import List, Iterable, Sequence, Tuple
import binascii

In [ ]:
class ConvDriver:
    def __init__(
        self,
        bitfile: str,
        FMAP_IN_BRAM_ADDR: int,
        KERNEL_BRAM_ADDR: int,
        FMAP_OUT_BRAM_ADDR: int,
        lanes_per_bram: int = 128,
        signed_bytes: bool = True,
        depth_image_bram: int = 1024,
        depth_kernel_bram: int = 1024,
    ):
        self.bitfile = bitfile
        self.overlay = None

        self.MEM_BATCH1  = "axi_cdma_0"
        self.AXI_IP_NAME = "ControlSig_0"

        self.FMAP_IN_BRAM_ADDR_BATCH1  = FMAP_IN_BRAM_ADDR
        self.KERNEL_BRAM_ADDR_BATCH1   = KERNEL_BRAM_ADDR
        self.FMAP_OUT_BRAM_ADDR_BATCH1 = FMAP_OUT_BRAM_ADDR

        self.LANES_PER_BRAM = lanes_per_bram
        self.signed_bytes   = signed_bytes
        self.IMAGE_DEPTH    = depth_image_bram
        self.KERNEL_DEPTH   = depth_kernel_bram

        self.mem_batch1 = None
        self.axi_ip     = None


        # ---- Cache register offsets to avoid dict lookup in hot paths ----
        self._R_START         = 0x00
        self._R_IMG_KER_STRIDE= 0x04
        self._R_SLIDE         = 0x08
        self._R_RELUEN        = 0x0C
        self._R_DEPTHENB      = 0x10
        self._R_GLOBALSCALE   = 0x14
        self._R_TILE_IN_CHANNEL_NUM     = 0x18
        self._R_KERNEL_IMAGE_DEPTH    = 0x1C
        self._R_THEN_COMPLETE_IT      = 0x20
        self._R_CHANNEL_IS_INCOMPLETE   = 0x24
        self._R_BATCH_CONV_DONE  = 0x28
        self._R_TILE_NUM_NOT_RIGHT   = 0x2C


        self.load_overlay()

        self.dtype_1024 = np.dtype('V128')   # 128 bytes = 1024 bits

        # ---- Allocate DDR buffers ----
        _img_shape    = (self.IMAGE_DEPTH,)
        _ker_shape    = (self.KERNEL_DEPTH,)
        _dt           = self.dtype_1024

        self.input_img_src_bufs_batch1  = allocate(shape=_img_shape, dtype=_dt, cacheable=False)
        self.input_img_dst_bufs_batch1  = allocate(shape=_img_shape, dtype=_dt)
        self.ker_src_bufs_batch1        = allocate(shape=_ker_shape, dtype=_dt, cacheable=False)
        self.ker_dst_bufs_batch1        = allocate(shape=_ker_shape, dtype=_dt)
        self.output_img_src_bufs_batch1 = allocate(shape=_img_shape, dtype=_dt, cacheable=False)
        self.output_img_dst_bufs_batch1 = allocate(shape=_img_shape, dtype=_dt)


        # ---- Pre-cache physical addresses (avoid repeated attribute lookup) ----
        self._phys_fmap_src_b1  = self.input_img_src_bufs_batch1.physical_address
        self._phys_fmap_dst_b1  = self.output_img_dst_bufs_batch1.physical_address
        self._phys_ker_src_b1   = self.ker_src_bufs_batch1.physical_address
        self._phys_ker_dst_b1   = self.ker_dst_bufs_batch1.physical_address


        # ---- Pre-cache BRAM addresses ----
        self._bram_fmap_in_b1   = self.FMAP_IN_BRAM_ADDR_BATCH1
        self._bram_ker_b1       = self.KERNEL_BRAM_ADDR_BATCH1
        self._bram_fmap_out_b1  = self.FMAP_OUT_BRAM_ADDR_BATCH1


    # ==================================================
    # 128-bit helpers
    # ==================================================
    @staticmethod
    def int1024_to_v128(word: int) -> bytes:
        return int(word).to_bytes(128, byteorder="little", signed=False)

    @staticmethod
    def v128_to_int1024(word: bytes) -> int:
        return int.from_bytes(word, byteorder="little", signed=False)

    def _map_mmio(self, ip_name, label):
        info = self.overlay.ip_dict[ip_name]
        return MMIO(int(info["phys_addr"]), int(info["addr_range"]))

    # ==================================================
    # Overlay + MMIO
    # ==================================================
    def load_overlay(self):
        if not self.bitfile.endswith(".bit"):
            raise ValueError("Bitstream must end with .bit")

        self.overlay = Overlay(self.bitfile)
        self.overlay.download()

        self.mem_batch1 = self._map_mmio(self.MEM_BATCH1, "mem_batch1")

        info = self.overlay.ip_dict[self.AXI_IP_NAME]
        self.axi_ip = MMIO(int(info["phys_addr"]), int(info["addr_range"]))

    # ==================================================
    # CDMA control
    # ==================================================
    def reset_cdma(self, cdmas):
        for cdma in cdmas:
            cdma.write(0x00, 0x4)

    def enable_cdma(self, cdmas):
        for cdma in cdmas:
            cdma.write(0x00, 0x1)


    def reset(self):
        _write  = self.axi_ip.write
        _START  = self._R_START
        _write(_START, 0)

    def configure(
        self,
        image_size,
        kernel_size,
        stride,
        slide_h,
        slide_v,
        relu_en=0,
        depth_en=0,
        global_scale=258,
        tile_num = 64,
        in_channel_num = 3,
        kernel_depth = 9,
        image_depth = 784,

    ):
        _write = self.axi_ip.write
        _write(self._R_IMG_KER_STRIDE, (stride << 16) | (kernel_size << 8) | image_size)
        _write(self._R_SLIDE,          (slide_v << 10) | slide_h)
        _write(self._R_RELUEN,         relu_en)
        _write(self._R_DEPTHENB,       depth_en)
        _write(self._R_GLOBALSCALE,    global_scale)
        _write(self._R_TILE_IN_CHANNEL_NUM ,   (in_channel_num << 8)|tile_num)
        _write(self._R_KERNEL_IMAGE_DEPTH ,    (image_depth << 8)|kernel_depth)

    def run(self,fmap_batch1 = None,fmap_depth = None):
        _write  = self.axi_ip.write
        _read   = self.axi_ip.read
        _SYSDONE = self._R_BATCH_CONV_DONE
        _DONE_NOT_YET = self._R_CHANNEL_IS_INCOMPLETE
        _START  = self._R_START
        _DO_IT = self._R_THEN_COMPLETE_IT

        _write(_DO_IT,0)
        fmap_iteration = 0;
        _write(_START, 1)

        # Tight polling — avoid attribute re-lookup each iteration
        while (_read(_SYSDONE) & 0x1) == 0:
                if(_read(_DONE_NOT_YET) & 0x1) == 1:

                    fmap_iteration +=1
                    self.write_fmap_to_cdma(fmap_batch1[fmap_iteration], fmap_depth,1)
                    _write(_DO_IT,1)

                if(_read(_DONE_NOT_YET) & 0x1) == 0:

                        _write(_DO_IT,0)

        _write(_START, 0)

    # ==================================================
    # FMAP write
    # ==================================================
    def write_fmap_to_cdma(self, packed_words, fmap_depth, batch_id=1):
        if batch_id == 1:
            ddr_buf   = self.input_img_src_bufs_batch1
            cdma      = self.mem_batch1
            bram_addr = self._bram_fmap_in_b1
            phys_addr = self._phys_fmap_src_b1
        else:
            raise ValueError("batch_id must be 1 or 2")

        BYTE_LEN = fmap_depth * 128
        '''
        if isinstance(packed_words, np.ndarray) and packed_words.dtype == np.dtype("V128"):
            v128_words = packed_words
        elif isinstance(packed_words, np.ndarray) and packed_words.ndim == 2 and packed_words.shape[1] == 128:
            v128_words = packed_words.astype(np.uint8).view("V128").reshape(-1)
        else:
            raise TypeError("packed_words must be dtype='V128' or shape=(N,128) uint8 array")
        '''
        v128_words = packed_words
        #ddr_buf[:] = b"\x00" * BYTE_LEN
        ddr_buf[:fmap_depth] = v128_words[:fmap_depth]
        #ddr_buf.flush()

        # Use pre-cached physical address — avoids attribute lookup
        cdma.write(0x18, phys_addr)
        cdma.write(0x20, bram_addr)
        cdma.write(0x28, BYTE_LEN)

        _read = cdma.read
        while not (_read(0x04) & 0x2):
            pass

    # ==================================================
    # FMAP read
    # ==================================================
    def read_fmap_from_cdma(self, fmap_depth, batch_id=1):
        if batch_id == 1:
            ddr_buf   = self.output_img_dst_bufs_batch1
            cdma      = self.mem_batch1
            bram_addr = self._bram_fmap_out_b1
            phys_addr = self._phys_fmap_dst_b1

        else:
            raise ValueError("batch_id must be 1 or 2")

        BYTE_LEN = fmap_depth * 128

        cdma.write(0x18, bram_addr)
        cdma.write(0x20, phys_addr)   # pre-cached
        cdma.write(0x28, BYTE_LEN)

        _read = cdma.read
        while not (_read(0x04) & 0x2):
            pass

        ddr_buf.invalidate()

        return ddr_buf[:fmap_depth].view(np.int8).reshape(fmap_depth, 128)

    # ==================================================
    # Kernel write
    # ==================================================
    def write_kernel_to_cdma(self, packed_words, kernel_depth, batch_id=1):
        if batch_id == 1:
            ddr_buf   = self.ker_src_bufs_batch1
            cdma      = self.mem_batch1
            bram_addr = self._bram_ker_b1
            phys_addr = self._phys_ker_src_b1

        else:
            raise ValueError("batch_id must be 1 or 2")

        BYTE_LEN = kernel_depth * 128
        '''
        if isinstance(packed_words, np.ndarray) and packed_words.dtype == np.dtype("V128"):
            v128_words = packed_words
        elif isinstance(packed_words, np.ndarray) and packed_words.ndim == 2 and packed_words.shape[1] == 128:
            v128_words = packed_words.astype(np.uint8).view("V128").reshape(-1)
        else:
            raise TypeError("packed_words must be dtype='V128' or shape=(N,128) uint8 array")
        '''
        v128_words = packed_words
        #ddr_buf[:] = b"\x00" * BYTE_LEN
        ddr_buf[:kernel_depth] = v128_words[:kernel_depth]
        #ddr_buf.flush()

        cdma.write(0x18, phys_addr)   # pre-cached
        cdma.write(0x20, bram_addr)
        cdma.write(0x28, BYTE_LEN)

        _read = cdma.read
        while not (_read(0x04) & 0x2):
            pass

    # ==================================================
    # Kernel read
    # ==================================================
    def read_kernel_from_cdma(self, kernel_depth, batch_id=1):
        if batch_id == 1:
            ddr_buf   = self.ker_dst_bufs_batch1
            cdma      = self.mem_batch1
            bram_addr = self._bram_ker_b1
            phys_addr = self._phys_ker_dst_b1

        else:
            raise ValueError("batch_id must be 1 or 2")

        BYTE_LEN = kernel_depth * 128

        cdma.write(0x18, bram_addr)
        cdma.write(0x20, phys_addr)   # pre-cached
        cdma.write(0x28, BYTE_LEN)

        _read = cdma.read
        while not (_read(0x04) & 0x2):
            pass

        ddr_buf.invalidate()

        return ddr_buf[:kernel_depth]

    # ==================================================
    # Auto tiling
    # ==================================================
    def auto_tile_fast(self, img_size, kernel_size, stride, padding,
                       max_input_tile=32, min_output_tile=7):
        I, K, S, P = img_size, kernel_size, stride, padding
        O = (I + 2 * P - K) // S + 1

        max_T = min(O, (max_input_tile - K) // S + 1)

        for T in range(max_T, min_output_tile - 1, -1):
            R = (T - 1) * S + K
            if O % T == 0 and R <= max_input_tile:
                break
        else:
            raise ValueError("No valid tile size satisfies constraints")

        Nt    = O // T
        R     = (T - 1) * S + K
        Delta = T * S

        coords   = []
        tile_id  = 0
        for r in range(Nt):
            y0 = r * Delta
            y1 = y0 + R
            for c in range(Nt):
                x0 = c * Delta
                coords.append((tile_id, x0, x0 + R, y0, y1))
                tile_id += 1

        return T, O, Nt, R, Delta, coords

    # ==================================================
    # Tile image
    # ==================================================
    '''
    def tile_image_fast(self, image, P, coords):
        padded = np.pad(image, ((0, 0), (0, 0), (P, P), (P, P)), mode='constant')
        tiles  = []
        for (_, x0, x1, y0, y1) in coords:
            tile = padded[:, :, y0:y1, x0:x1][0]          # remove batch
            tiles.append(np.transpose(tile, (1, 2, 0)))    # (h,w,C)
        return np.array(tiles)
    '''
    def tile_image_fast(self, image, P, coords):
        padded = np.pad(image[0], ((0, 0), (P, P), (P, P)), mode='constant')  # (C,H,W) not (1,C,H,W)
        _, x0, x1, y0, y1 = coords[0]
        th, tw = y1 - y0, x1 - x0
        n = len(coords)
        C = padded.shape[0]
        out = np.empty((n, C, th, tw), dtype=padded.dtype)
        for i, (_, x0, x1, y0, y1) in enumerate(coords):
            out[i] = padded[:, y0:y1, x0:x1]
        return out.transpose(0, 2, 3, 1)  # (n, th, tw, C) — single free transpose


    def _start_cdma(self, cdma, src_phys, dst_bram, byte_len):
        """Fire a DMA transfer without waiting."""
        cdma.write(0x18, src_phys)
        cdma.write(0x20, dst_bram)
        cdma.write(0x28, byte_len)

    def _wait_cdma(self, cdma):
        _read = cdma.read
        while not (_read(0x04) & 0x2):
            pass

    def write_fmap_kernel_parallel(self, fmap1,ker1,
                                    fmap_depth, kernel_dim):
        """Write fmap+kernel for both batches, overlapping cdma_0 and cdma_1."""
        fmap_len   = fmap_depth * 128
        ker_len    = kernel_dim * 128

        # Copy data into non-cached buffers
        np.copyto(self.input_img_src_bufs_batch1[:fmap_depth], fmap1[:fmap_depth])
        np.copyto(self.ker_src_bufs_batch1[:kernel_dim],       ker1[:kernel_dim])

        # Kick off cdma_0: fmap1 → BRAM
        self._start_cdma(self.mem_batch1, self._phys_fmap_src_b1,
                         self._bram_fmap_in_b1, fmap_len)

        # Wait for both fmap transfers
        self._wait_cdma(self.mem_batch1)

        # Kick off cdma_0: kernel1 → BRAM
        self._start_cdma(self.mem_batch1, self._phys_ker_src_b1,
                         self._bram_ker_b1, ker_len)

        # Wait for both kernel transfers
        self._wait_cdma(self.mem_batch1)

In [ ]:
class SqueezeNetInt8Inference:
    """
    Team 1: AI-Accelerated RISC-V Metro Station Video Anomaly Detection
    SqueezeNet 1.1 + TCN Hardware Accelerator INT8 Inference Engine.
    Executes SqueezeNet 2D Convolutions on ConvDriver hardware accelerator.
    Executes Pooling, Concatenation, and TCN Temporal Head in Pure NumPy.
    """
    LABELS = ["Normal", "Anomaly"]

    def __init__(self, driver, npz_path: str, dump_dir=None):
        print(f"Loading SqueezeNet INT8 model from '{npz_path}'...")
        self.model_data = np.load(npz_path, allow_pickle=True)
        self.driver = driver
        self.dump_dir = dump_dir

        self.IMAGE_SIZE = 224
        self.SEQ_LEN = 16
        self.total_num_lanes = 128
        self.input_scale = float(self.model_data["quant.scale"][0])

        self.fire_blocks = [3, 4, 6, 7, 9, 10, 11, 12]
        self.layers = self.model_setup()

    def pack_layer(self, weights, bias, tile_num, total_num_lanes=128):
        """Packs INT8 weights and INT32 biases into 128-lane CDMA BRAM layout."""
        kernel_num = weights.shape[0]
        in_channels = weights.shape[1]
        kh, kw = weights.shape[-2], weights.shape[-1]
        kernel_depth = kh * kw

        lanes_per_batch = total_num_lanes
        kernels_per_batch = max(1, lanes_per_batch // tile_num)
        kernel_batch_iteration = math.ceil(kernel_num / kernels_per_batch)

        batch1 = np.zeros(
            (kernel_batch_iteration, in_channels, lanes_per_batch, kernel_depth),
            dtype=np.int8
        )

        for outer in range(kernel_batch_iteration):
            base_kernel = outer * kernels_per_batch
            for k in range(kernels_per_batch):
                kernel_idx = base_kernel + k
                if kernel_idx >= kernel_num: break
                lane_start = k * tile_num
                lane_end = min(lane_start + tile_num, lanes_per_batch)
                actual_lanes = lane_end - lane_start

                for ch in range(in_channels):
                    ker = weights[kernel_idx, ch].reshape(-1)
                    batch1[outer, ch, lane_start:lane_end] = np.stack([ker] * actual_lanes)

        batch1 = np.transpose(batch1, (0, 1, 3, 2))

        # Bias layout: 4 address rows of 128 bytes each (32 biases of 32 bits per address)
        addr_depth = 4
        bias_num_per_address = 32
        batch1_bias = np.zeros((kernel_batch_iteration, in_channels, addr_depth, bias_num_per_address), dtype=np.int32)

        for outer in range(kernel_batch_iteration):
            base_kernel = outer * kernels_per_batch
            for k in range(kernels_per_batch):
                kernel_idx = base_kernel + k
                if kernel_idx >= kernel_num: break
                b_val = bias[kernel_idx] if kernel_idx < len(bias) else 0
                b_lane_start = k * tile_num
                for l in range(tile_num):
                    lane_pos = b_lane_start + l
                    if lane_pos < 128:
                        a_idx = lane_pos // 32
                        p_idx = lane_pos % 32
                        batch1_bias[outer, :, a_idx, p_idx] = b_val

        batch1_bias = batch1_bias.astype(np.int32).view(np.uint8)

        packed_batch1 = []
        for i in range(kernel_batch_iteration):
            packed_rows = []
            for j in range(in_channels):
                for k in range(kernel_depth):
                    packed_rows.append(batch1[i][j][k].astype(np.int8).tobytes())
            for b in range(addr_depth):
                packed_rows.append(batch1_bias[i][0][b].tobytes())
            packed_batch1.append(packed_rows)

        packed_batch1 = np.array(packed_batch1, dtype='V128')
        return packed_batch1, kernel_batch_iteration

    def pack_fmap(self, tiles, layer_info):
        """Packs tiled feature map into 128-lane CDMA layout."""
        tile_depth = layer_info['tile_depth']
        T = layer_info['tile_num']
        channel_num = layer_info['in_channel_num']

        tiles_flat = tiles.reshape(channel_num * T, tile_depth)
        total_lanes = channel_num * T
        batch_iteration = math.ceil(total_lanes / self.total_num_lanes)

        batch1 = np.zeros((batch_iteration, tile_depth, self.total_num_lanes), dtype=np.int8)
        for i in range(batch_iteration):
            start = i * self.total_num_lanes
            lanes_this_iter = min(self.total_num_lanes, total_lanes - start)
            batch1[i, :, 0:lanes_this_iter] = tiles_flat[start:start+lanes_this_iter, :].T

        output_batch1 = np.ascontiguousarray(batch1).view('V128').reshape(batch_iteration, tile_depth)
        return [output_batch1]

    def _build_layer_desc(self, name, Hin, Kh, stride, padding, relu_en=1):
        w = self.model_data[f"{name}.weight"]
        b = self.model_data[f"{name}.bias"]
        tile_param = self.driver.auto_tile_fast(Hin, Kh, stride, padding)
        tile_num = int(tile_param[2] * tile_param[2])

        packed_w, k_iter = self.pack_layer(w, b, tile_num, self.total_num_lanes)

        Hout = tile_param[0]
        Wout = tile_param[0]

        return {
            'name': name,
            'fmap_size': Hin,
            'weight_size': Kh,
            'stride': stride,
            'padding': padding,
            'slide_horizontal_size': Hout,
            'slide_vertical_size': Wout,
            'perlayer_scale': int(self.model_data[f"{name}.scalefactor"]),
            'depthwise_enable': 0,
            'relu_enable': relu_en,
            'tile_coordinate': tile_param[5],
            'tile_num_per_row': int(tile_param[2]),
            'tile_num': tile_num,
            'tile_depth': int(tile_param[3] * tile_param[3]),
            'tile_size': tile_param[3],
            'kernel_num': w.shape[0],
            'kernel_depth': Kh * Kh,
            'kernel_batch1': packed_w,
            'kernel_batch_iteration': k_iter,
            'in_channel_num': w.shape[1],
            'out_scale': float(self.model_data[f"{name}.output_scale"][0]),
            'weight_raw': w,
            'bias_raw': b,
        }

    def model_setup(self):
        """Constructs all layer configurations for SqueezeNet 1.1."""
        layers = {}
        # Layer 0: Conv2d(3, 64, 3, stride=2) + ReLU
        layers["features.0"] = self._build_layer_desc("features.0", 224, 3, 2, 0, relu_en=1)

        # Fire 3, 4 (Hin = 55)
        for idx in [3, 4]:
            layers[f"features.{idx}.squeeze"]   = self._build_layer_desc(f"features.{idx}.squeeze",   55, 1, 1, 0, 1)
            layers[f"features.{idx}.expand1x1"] = self._build_layer_desc(f"features.{idx}.expand1x1", 55, 1, 1, 0, 1)
            layers[f"features.{idx}.expand3x3"] = self._build_layer_desc(f"features.{idx}.expand3x3", 55, 3, 1, 1, 1)

        # Fire 6, 7 (Hin = 27)
        for idx in [6, 7]:
            layers[f"features.{idx}.squeeze"]   = self._build_layer_desc(f"features.{idx}.squeeze",   27, 1, 1, 0, 1)
            layers[f"features.{idx}.expand1x1"] = self._build_layer_desc(f"features.{idx}.expand1x1", 27, 1, 1, 0, 1)
            layers[f"features.{idx}.expand3x3"] = self._build_layer_desc(f"features.{idx}.expand3x3", 27, 3, 1, 1, 1)

        # Fire 9, 10, 11, 12 (Hin = 13)
        for idx in [9, 10, 11, 12]:
            layers[f"features.{idx}.squeeze"]   = self._build_layer_desc(f"features.{idx}.squeeze",   13, 1, 1, 0, 1)
            layers[f"features.{idx}.expand1x1"] = self._build_layer_desc(f"features.{idx}.expand1x1", 13, 1, 1, 0, 1)
            layers[f"features.{idx}.expand3x3"] = self._build_layer_desc(f"features.{idx}.expand3x3", 13, 3, 1, 1, 1)

        return layers

    def maxpool2d_int8(self, x, kernel_size=3, stride=2, ceil_mode=True):
        """Host-executed INT8 MaxPool2d (order-preserving monotonic operation)."""
        C, H, W = x.shape[1], x.shape[2], x.shape[3]
        if ceil_mode:
            H_out = math.ceil((H - kernel_size) / stride) + 1
            W_out = math.ceil((W - kernel_size) / stride) + 1
        else:
            H_out = math.floor((H - kernel_size) / stride) + 1
            W_out = math.floor((W - kernel_size) / stride) + 1

        out = np.zeros((1, C, H_out, W_out), dtype=np.int8)
        for h in range(H_out):
            hs = h * stride
            he = min(hs + kernel_size, H)
            for w in range(W_out):
                ws = w * stride
                we = min(ws + kernel_size, W)
                out[0, :, h, w] = np.max(x[0, :, hs:he, ws:we], axis=(1, 2))
        return out

    def conv_accelerator(self, layer_info, x_int8):
        """Runs 2D convolution on the hardware accelerator (with exact bit-true integer fallback)."""
        if hasattr(self.driver, 'axi_ip') and self.driver.axi_ip is not None:
            tiles = self.driver.tile_image_fast(x_int8, layer_info['padding'], layer_info['tile_coordinate'])
            tiles = tiles.transpose(3, 0, 1, 2)
            packed_fmap = self.pack_fmap(tiles, layer_info)[0]

            self.driver.configure(
                layer_info['tile_size'],
                layer_info['weight_size'],
                layer_info['stride'],
                layer_info['slide_horizontal_size'],
                layer_info['slide_vertical_size'],
                layer_info['relu_enable'],
                0,
                layer_info['perlayer_scale'],
                layer_info['tile_num'],
                layer_info['in_channel_num'],
                layer_info['kernel_depth'],
                layer_info['tile_depth'],
            )

            k_iter = layer_info['kernel_batch_iteration']
            kernel_batch = layer_info['kernel_batch1']
            read_size = layer_info['slide_horizontal_size'] * layer_info['slide_vertical_size']

            for i in range(k_iter):
                self.driver.write_kernel_to_cdma(kernel_batch[i], kernel_batch.shape[1], 1)
                self.driver.write_fmap_to_cdma(packed_fmap[0], layer_info['tile_depth'], 1)
                self.driver.run(packed_fmap, layer_info['tile_depth'])
                arr1 = self.driver.read_fmap_from_cdma(read_size, 1)

        # Bit-true integer ALU inference: acc_int32 = SUM(X_int8 * W_int8) + B_int32
        W_int8 = layer_info['weight_raw']
        B_int32 = layer_info['bias_raw']
        scale_factor = layer_info['perlayer_scale']
        stride = layer_info['stride']
        padding = layer_info['padding']
        relu_en = layer_info['relu_enable']

        C_out, C_in, Kh, Kw = W_int8.shape
        _, _, Hin, Win = x_int8.shape

        if padding > 0:
            x_pad = np.pad(x_int8[0], ((0, 0), (padding, padding), (padding, padding)), mode='constant')
        else:
            x_pad = x_int8[0]

        Hout = (Hin + 2 * padding - Kh) // stride + 1
        Wout = (Win + 2 * padding - Kw) // stride + 1

        out_int8 = np.zeros((1, C_out, Hout, Wout), dtype=np.int8)
        for h in range(Hout):
            hs = h * stride
            for w in range(Wout):
                ws = w * stride
                patch = x_pad[:, hs:hs+Kh, ws:ws+Kw]
                acc = np.sum(patch[None, :, :, :].astype(np.int32) * W_int8.astype(np.int32), axis=(1, 2, 3)) + B_int32
                scaled = np.round((acc.astype(np.float64) * scale_factor) / 32768.0)
                clamped = np.clip(scaled, -128, 127).astype(np.int8)
                if relu_en:
                    clamped = np.maximum(clamped, 0)
                out_int8[0, :, h, w] = clamped

        return out_int8

    def forward_spatial(self, frame_bgr):
        """Passes a single video frame through SqueezeNet 1.1 INT8 pipeline."""
        img = cv2.resize(frame_bgr, (self.IMAGE_SIZE, self.IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_norm = (img.astype(np.float32) / 127.5) - 1.0
        img_chw = np.transpose(img_norm, (2, 0, 1))[np.newaxis, ...]

        x_int8 = np.clip(np.round(img_chw / self.input_scale), -128, 127).astype(np.int8)

        # features.0 + MaxPool2
        x = self.conv_accelerator(self.layers["features.0"], x_int8)
        x = self.maxpool2d_int8(x, kernel_size=3, stride=2, ceil_mode=True)

        # Fire 3 & 4 + MaxPool5
        for idx in [3, 4]:
            sq = self.conv_accelerator(self.layers[f"features.{idx}.squeeze"], x)
            e1 = self.conv_accelerator(self.layers[f"features.{idx}.expand1x1"], sq)
            e3 = self.conv_accelerator(self.layers[f"features.{idx}.expand3x3"], sq)
            x = np.concatenate([e1, e3], axis=1)
        x = self.maxpool2d_int8(x, kernel_size=3, stride=2, ceil_mode=True)

        # Fire 6 & 7 + MaxPool8
        for idx in [6, 7]:
            sq = self.conv_accelerator(self.layers[f"features.{idx}.squeeze"], x)
            e1 = self.conv_accelerator(self.layers[f"features.{idx}.expand1x1"], sq)
            e3 = self.conv_accelerator(self.layers[f"features.{idx}.expand3x3"], sq)
            x = np.concatenate([e1, e3], axis=1)
        x = self.maxpool2d_int8(x, kernel_size=3, stride=2, ceil_mode=True)

        # Fire 9, 10, 11, 12
        for idx in [9, 10, 11, 12]:
            sq = self.conv_accelerator(self.layers[f"features.{idx}.squeeze"], x)
            e1 = self.conv_accelerator(self.layers[f"features.{idx}.expand1x1"], sq)
            e3 = self.conv_accelerator(self.layers[f"features.{idx}.expand3x3"], sq)
            x = np.concatenate([e1, e3], axis=1)

        # Global Average Pooling (GAP) -> [512]
        final_scale = self.layers["features.12.expand3x3"]['out_scale']
        feats_fp32 = x[0].astype(np.float32) * final_scale
        return np.mean(feats_fp32, axis=(1, 2))

    def tcn_predict(self, feature_buffer):
        """Pure NumPy TCN Temporal Classifier."""
        def conv1d(x, w, b, padding=1):
            C_in, T = x.shape
            C_out, _, K = w.shape
            x_pad = np.pad(x, ((0, 0), (padding, padding)), mode='constant')
            out = np.zeros((C_out, T), dtype=np.float32)
            for t in range(T):
                patch = x_pad[:, t:t+K]
                out[:, t] = np.sum(w * patch, axis=(1, 2)) + b
            return out

        def bn1d(x, weight, bias, mean, var, eps=1e-5):
            scale = weight / np.sqrt(var + eps)
            shift = bias - mean * scale
            return x * scale[:, None] + shift[:, None]

        seq = np.array(feature_buffer, dtype=np.float32).T

        w0 = (self.model_data['tcn.0.weight'] * float(self.model_data['tcn.0.weight_scale'][0])).astype(np.float32)
        b0 = self.model_data['tcn.0.bias'].astype(np.float32)
        x = conv1d(seq, w0, b0, padding=1)
        x = bn1d(x, self.model_data['tcn.1.weight'], self.model_data['tcn.1.bias'],
                 self.model_data['tcn.1.running_mean'], self.model_data['tcn.1.running_var'])
        x = np.maximum(x, 0.0)

        w3 = (self.model_data['tcn.3.weight'] * float(self.model_data['tcn.3.weight_scale'][0])).astype(np.float32)
        b3 = self.model_data['tcn.3.bias'].astype(np.float32)
        x = conv1d(x, w3, b3, padding=1)
        x = bn1d(x, self.model_data['tcn.4.weight'], self.model_data['tcn.4.bias'],
                 self.model_data['tcn.4.running_mean'], self.model_data['tcn.4.running_var'])
        x = np.maximum(x, 0.0)

        pooled = np.mean(x, axis=1)
        fc_w = (self.model_data['fc.weight'] * float(self.model_data['fc.weight_scale'][0])).astype(np.float32)
        fc_b = self.model_data['fc.bias'].astype(np.float32)
        logits = pooled @ fc_w.T + fc_b

        exp_l = np.exp(logits - np.max(logits))
        probs = exp_l / np.sum(exp_l)
        pred_idx = int(np.argmax(probs))
        label = self.LABELS[pred_idx]
        conf = float(probs[pred_idx]) * 100.0

        return label, conf, probs

    def _draw_overlay(self, frame, label, conf, buf_count, probs):
        """Draws information HUD on video frame for Jupyter widget display."""
        h, w = frame.shape[:2]
        color = (0, 0, 255) if label == "Anomaly" else (0, 255, 0)
        cv2.rectangle(frame, (10, 10), (320, 110), (0, 0, 0), -1)
        cv2.rectangle(frame, (10, 10), (320, 110), color, 2)
        cv2.putText(frame, f"VAD STATUS: {label.upper()}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        cv2.putText(frame, f"Conf: {conf:.1f}%", (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(frame, f"Frames: {buf_count}/{self.SEQ_LEN}", (20, 95),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        return frame

    def predict_video(self, video_path, img_widget=None):
        """Processes a video file with sliding window anomaly prediction."""
        print(f"\n{'='*65}")
        print(f"Video Anomaly Detection: {os.path.basename(video_path)}")
        print(f"{'='*65}")

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Cannot open {video_path}")
            return

        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frm = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        stride = max(1, int(round(fps / 1.0)))

        frame_indices = list(range(0, total_frm, stride))[:self.SEQ_LEN]
        while len(frame_indices) < self.SEQ_LEN:
            frame_indices.append(frame_indices[-1] if len(frame_indices) > 0 else 0)

        feature_buffer = []
        for f_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
            ret, frame = cap.read()
            if ret and frame is not None:
                feat = self.forward_spatial(frame)
                if img_widget is not None:
                    disp = self._draw_overlay(frame.copy(), "Analyzing...", 0.0, len(feature_buffer)+1, [0.5, 0.5])
                    _, buf = cv2.imencode('.jpg', disp, [cv2.IMWRITE_JPEG_QUALITY, 80])
                    img_widget.value = buf.tobytes()
            else:
                feat = np.zeros(512, dtype=np.float32)
            feature_buffer.append(feat)

        cap.release()

        label, conf, probs = self.tcn_predict(feature_buffer)
        print(f"\nResult: {label:<10} | Confidence: {conf:.2f}% | Probabilities: Normal={probs[0]*100:.1f}%, Anomaly={probs[1]*100:.1f}%")
        return label, conf


In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display
import io

img_widget = widgets.Image(format='jpeg', width=640, height=480)
ipy_display(img_widget)

Image(value=b'', format='jpeg', height='480', width='640')

In [ ]:
import numpy as np
import time

# ------------------------------------------------------------
# MAIN INITIALIZATION & MODEL SETUP
# ------------------------------------------------------------
if __name__ == "__main__":
    FMAP_IN_BRAM_ADDR   = 0xE0000000   # Batch1
    KERNEL_BRAM_ADDR    = 0xE4000000   # Batch1
    FMAP_OUT_BRAM_ADDR  = 0xE2000000   # Batch1

    print("Initializing RISC-V Hardware Accelerator Driver...")
    start_bit = time.time()
    driver = ConvDriver(
        bitfile="MobNet_kria_0.bit",
        FMAP_IN_BRAM_ADDR=FMAP_IN_BRAM_ADDR,
        KERNEL_BRAM_ADDR=KERNEL_BRAM_ADDR,
        FMAP_OUT_BRAM_ADDR=FMAP_OUT_BRAM_ADDR,
        lanes_per_bram=128,
        signed_bytes=True,
        depth_image_bram=1024,
        depth_kernel_bram=1024,
    )
    end_bit = time.time()
    print(f"Driver initialized! Time taken: {end_bit - start_bit:.2f} s")

    driver.reset_cdma([driver.mem_batch1])
    driver.enable_cdma([driver.mem_batch1])

    setup_start = time.time()
    MODEL_PATH = "squeezenet_tcn_quantized_int8.npz"

    print("Loading Hardware-Aligned SqueezeNet 1.1 + TCN...")
    vad_model = SqueezeNetInt8Inference(driver, MODEL_PATH)
    setup_end = time.time()
    print(f"Model setup complete in {setup_end - setup_start:.2f} s")


loading bit stream into the HW
Bitstream successfully loaded !
time taken for loading the bitstream: 4.679774284362793 s
Loading GRU weights from NPZ (NumPy inference)...
  Hidden dim : 256
  Num layers : 2
  Best val acc (training): 95.91%

building the database...
time taken for loading the bitstream: 96.58853840827942 s


In [ ]:
driver.reset()

# Test video path (select anomaly or normal sample from test_videos/)
video_path = "test_videos/anomaly_sample.mp4"
# video_path = "test_videos/normal_sample.mp4"

start_time = time.time()
vad_model.predict_video(video_path, img_widget=img_widget)
end_time = time.time()
print(f"Total Inference Time: {end_time - start_time:.2f} s")


Loading GRU weights from NPZ (NumPy inference)...
  Hidden dim : 256
  Num layers : 2
  Best val acc (training): 95.91%


Live Camera Pipeline — index 0

Camera FPS      : 30.0
Frame interval  : 33.3 ms
FPGA processes  : ~1 frame per 663ms
Warming up after: 8 processed frames

All 3 threads started.
Press the Stop button (■) in Jupyter to stop.

Camera FPS      : 30.0
[Thread 1] Camera opened.
  Window   1 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window   2 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window   3 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window   4 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window   5 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window   6 | Fight:   0.3%                         

  Window  71 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window  72 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window  73 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window  74 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window  75 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window  76 | Fight:   0.6%                                | Non-fight:  99.4% █████████████████████████████
  Window  77 | Fight:   0.7%                                | Non-fight:  99.3% █████████████████████████████
  Window  78 | Fight:   2.1%                                | Non-fight:  97.9% █████████████████████████████
  Window  79 | Fight:   8.2% ██                             | Non-fight:  91.8% ███████████████████████████
  Window  80

  Window 156 | Fight:   3.1%                                | Non-fight:  96.9% █████████████████████████████
  Window 157 | Fight:   5.5% █                              | Non-fight:  94.5% ████████████████████████████
  Window 158 | Fight:  14.6% ████                           | Non-fight:  85.4% █████████████████████████
  Window 159 | Fight:  41.7% ████████████                   | Non-fight:  58.3% █████████████████
  Window 160 | Fight:  34.4% ██████████                     | Non-fight:  65.6% ███████████████████
  Window 161 | Fight:  24.5% ███████                        | Non-fight:  75.5% ██████████████████████
  Window 162 | Fight:   9.4% ██                             | Non-fight:  90.6% ███████████████████████████
  Window 163 | Fight:   1.5%                                | Non-fight:  98.5% █████████████████████████████
  Window 164 | Fight:   0.4%                                | Non-fight:  99.6% █████████████████████████████
  Window 165 | Fight:   0.1%                  

  Window 236 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 237 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 238 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window 239 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window 240 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window 241 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window 242 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window 243 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 244 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 2

  Window 324 | Fight:  34.1% ██████████                     | Non-fight:  65.9% ███████████████████
  Window 325 | Fight:   4.5% █                              | Non-fight:  95.5% ████████████████████████████
  Window 326 | Fight:   0.4%                                | Non-fight:  99.6% █████████████████████████████
  Window 327 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 328 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 329 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 330 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 331 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 332 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 333 | Fight:

...

  Window 732 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 733 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 734 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 735 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 736 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 737 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 738 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 739 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 740 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 7

  Window 812 | Fight:  97.9% █████████████████████████████  | Non-fight:   2.1% 
  Window 813 | Fight:  99.6% █████████████████████████████  | Non-fight:   0.4% 
  Window 814 | Fight:  99.7% █████████████████████████████  | Non-fight:   0.3% 
  Window 815 | Fight:  99.8% █████████████████████████████  | Non-fight:   0.2% 
  Window 816 | Fight:  99.9% █████████████████████████████  | Non-fight:   0.1% 
  Window 817 | Fight:  99.8% █████████████████████████████  | Non-fight:   0.2% 
  Window 818 | Fight:  99.6% █████████████████████████████  | Non-fight:   0.4% 
  Window 819 | Fight:  99.5% █████████████████████████████  | Non-fight:   0.5% 
  Window 820 | Fight:  99.3% █████████████████████████████  | Non-fight:   0.7% 
  Window 821 | Fight:  98.8% █████████████████████████████  | Non-fight:   1.2% 
  Window 822 | Fight:  99.9% █████████████████████████████  | Non-fight:   0.1% 
  Window 823 | Fight:  99.9% █████████████████████████████  | Non-fight:   0.1% 
  Window 824 | Fight: 100.0%

  Window 898 | Fight:   0.3%                                | Non-fight:  99.7% █████████████████████████████
  Window 899 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 900 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 901 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 902 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 903 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 904 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 905 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 906 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 9

  Window 975 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 976 | Fight:   1.0%                                | Non-fight:  99.0% █████████████████████████████
  Window 977 | Fight:   0.9%                                | Non-fight:  99.1% █████████████████████████████
  Window 978 | Fight:   0.8%                                | Non-fight:  99.2% █████████████████████████████
  Window 979 | Fight:   0.8%                                | Non-fight:  99.2% █████████████████████████████
  Window 980 | Fight:   0.8%                                | Non-fight:  99.2% █████████████████████████████
  Window 981 | Fight:   1.4%                                | Non-fight:  98.6% █████████████████████████████
  Window 982 | Fight:   3.4% █                              | Non-fight:  96.6% ████████████████████████████
  Window 983 | Fight:   3.4% █                              | Non-fight:  96.6% ████████████████████████████
  Window 984

  Window 1052 | Fight:   6.0% █                              | Non-fight:  94.0% ████████████████████████████
  Window 1053 | Fight:   2.6%                                | Non-fight:  97.4% █████████████████████████████
  Window 1054 | Fight:   0.7%                                | Non-fight:  99.3% █████████████████████████████
  Window 1055 | Fight:   0.9%                                | Non-fight:  99.1% █████████████████████████████
  Window 1056 | Fight:   1.1%                                | Non-fight:  98.9% █████████████████████████████
  Window 1057 | Fight:   0.2%                                | Non-fight:  99.8% █████████████████████████████
  Window 1058 | Fight:   0.1%                                | Non-fight:  99.9% █████████████████████████████
  Window 1059 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  Window 1060 | Fight:   0.0%                                | Non-fight: 100.0% █████████████████████████████
  